In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
  output_df = input_df.withColumn("ingestion_date", current_timestamp())
  return output_df

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import DataFrame
from delta.tables import DeltaTable

def merge_delta_data(
    input_df: DataFrame,
    delta_table_path: str,
    merge_condition: str,
    partition_columns: list = None,
    file_date_value: str = None, 
    file_date_column: str = "file_date",
):
  
  # Check if the table exists
  table_exists = DeltaTable.isDeltaTable(spark, delta_table_path)

# Check if we have already loaded this data (merge breaks if we have)
  already_loaded = (
      spark.read \
        .format("delta") \
        .load(delta_table_path) \
        .filter(col(file_date_column) == file_date_value) \
        .limit(1)
        .count() > 0
    )
  
  # Do nothing if we have already loaded it (also saves time)
  if already_loaded:
      print(f"Skipping merge: {file_date_value} already in {delta_table_path}")
      return
          
  # If the table exists then merge
  if table_exists:
      delta_table = DeltaTable.forPath(spark, delta_table_path)
      delta_table = delta_table.alias("tgt") \
        .merge(input_df.alias("src"), merge_condition) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

  # If it does not, write the data to the path
  else:

    # Partition if selected
    if partition_columns:
      input_df.write.mode("overwrite") \
      .format("delta") \
      .partitionBy(*partition_columns) \
      .save(delta_table_path)

    # No partition if not selected
    else:
      input_df.write.mode("overwrite") \
      .format("delta") \
      .save(delta_table_path)
